In [1]:
import os
import cv2
import time
import glob
import numpy as np
from pathlib import Path
from sklearn.cluster import KMeans
from skimage.segmentation import slic
from skimage.feature import local_binary_pattern
from sklearn.metrics import precision_score, recall_score, f1_score, jaccard_score

DATASET_ROOT = r"A:\wendang\comp9517\EWS-Dataset"
OUTPUT_ROOT = r"A:\wendang\comp9517\results\traditional"

N_SEGMENTS = 300
COMPACTNESS = 10
SIGMA = 1.0
LBP_RADIUS = 1
LBP_POINTS = 8 * LBP_RADIUS
LBP_METHOD = "uniform"
MORPH_KERNEL_SIZE = 5
MIN_COMPONENT_AREA = 80
SEEDS = [0, 1, 2, 3, 4]

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def read_img(p):
    img = cv2.imread(p, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(p)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def read_mask(p):
    mask = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(p)
    return (mask > 127).astype(np.uint8)

def pair(folder):
    files = glob.glob(os.path.join(folder, "*.png"))
    imgs = []
    masks = {}
    for f in files:
        name = Path(f).stem
        if name.endswith("_mask"):
            masks[name[:-5]] = f
        else:
            imgs.append((name, f))
    pairs = [(img, masks[name]) for name, img in imgs if name in masks]
    if len(pairs) == 0:
        raise RuntimeError(f"No pairs found in {folder}")
    return sorted(pairs, key=lambda x: x[0])

def get_data():
    train = pair(os.path.join(DATASET_ROOT, "train"))
    val = pair(os.path.join(DATASET_ROOT, "validation"))
    test = pair(os.path.join(DATASET_ROOT, "test"))
    return train, val, test

def exg(img):
    x = img.astype(np.float32) / 255.0
    return 2.0 * x[:, :, 1] - x[:, :, 0] - x[:, :, 2]

def features(img, seg):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    lbp = local_binary_pattern(gray, LBP_POINTS, LBP_RADIUS, method=LBP_METHOD)
    e = exg(img)
    n = int(seg.max()) + 1
    f = np.zeros((n, 5), dtype=np.float32)
    for i in range(n):
        m = (seg == i)
        if np.sum(m) == 0:
            continue
        r = img[:, :, 0][m].astype(np.float32)
        g = img[:, :, 1][m].astype(np.float32)
        b = img[:, :, 2][m].astype(np.float32)
        ee = e[m].astype(np.float32)
        ll = lbp[m].astype(np.float32)
        f[i, 0] = np.mean(r)
        f[i, 1] = np.mean(g)
        f[i, 2] = np.mean(b)
        f[i, 3] = np.mean(ee)
        f[i, 4] = np.mean(ll)
    f = np.nan_to_num(f, nan=0.0, posinf=0.0, neginf=0.0)
    mu = np.mean(f, axis=0, keepdims=True)
    sd = np.std(f, axis=0, keepdims=True)
    sd[sd < 1e-6] = 1.0
    f = (f - mu) / sd
    f = np.nan_to_num(f, nan=0.0, posinf=0.0, neginf=0.0)
    return f

def post(m):
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (MORPH_KERNEL_SIZE, MORPH_KERNEL_SIZE))
    m = (m * 255).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, k)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    cleaned = np.zeros_like(m)
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if area >= MIN_COMPONENT_AREA:
            cleaned[labels == i] = 255
    return (cleaned > 127).astype(np.uint8)

def fallback_segment(img):
    e = exg(img)
    thr = np.mean(e)
    pred = (e > thr).astype(np.uint8)
    return post(pred)

def segment(img, seed):
    rgb = img.astype(np.float32) / 255.0
    seg = slic(
        rgb,
        n_segments=N_SEGMENTS,
        compactness=COMPACTNESS,
        sigma=SIGMA,
        start_label=0
    )

    n_regions = int(seg.max()) + 1
    if n_regions < 2:
        return fallback_segment(img)

    f = features(img, seg)
    if f.shape[0] < 2:
        return fallback_segment(img)

    km = KMeans(n_clusters=2, random_state=seed, n_init=10)
    lab = km.fit_predict(f)

    e = exg(img)
    scores = []
    for i in range(2):
        ids = np.where(lab == i)[0]
        if len(ids) == 0:
            scores.append(-1e9)
            continue
        mm = np.isin(seg, ids)
        if np.sum(mm) == 0:
            scores.append(-1e9)
            continue
        scores.append(float(np.mean(e[mm])))

    crop = int(np.argmax(scores))
    pred = np.zeros(seg.shape, dtype=np.uint8)
    crop_ids = np.where(lab == crop)[0]
    for i in crop_ids:
        pred[seg == i] = 1
    return post(pred)

def evaluate(gt, pred):
    g = gt.flatten()
    p = pred.flatten()
    return [
        precision_score(g, p, zero_division=0),
        recall_score(g, p, zero_division=0),
        f1_score(g, p, zero_division=0),
        jaccard_score(g, p, zero_division=0)
    ]

def distort(img, mode):
    if mode == "noise":
        noise = np.random.normal(0, 20, img.shape)
        return np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    if mode == "blur":
        return cv2.GaussianBlur(img, (5, 5), 0)
    if mode == "dark":
        return np.clip(img.astype(np.float32) * 0.5, 0, 255).astype(np.uint8)
    return img

def overlay(img, mask):
    out = img.copy()
    green = np.zeros_like(out)
    green[:, :, 1] = 255
    alpha = 0.4
    out[mask == 1] = (alpha * green[mask == 1] + (1 - alpha) * out[mask == 1]).astype(np.uint8)
    return out

def run(pairs, mode=None, save_dir=None, save_images=False):
    if save_dir is not None:
        ensure_dir(save_dir)

    all_metrics = []
    for seed in SEEDS:
        seed_metrics = []
        np.random.seed(seed)

        for img_p, mask_p in pairs:
            img = read_img(img_p)
            if mode is not None:
                img = distort(img, mode)
            gt = read_mask(mask_p)

            t = time.time()
            pred = segment(img, seed)
            dt = time.time() - t

            p, r, f, i = evaluate(gt, pred)
            seed_metrics.append([p, r, f, i, dt])

            if save_images and save_dir is not None and seed == SEEDS[0]:
                base = Path(img_p).stem
                cv2.imwrite(os.path.join(save_dir, base + "_pred.png"), (pred * 255).astype(np.uint8))
                ov = overlay(img, pred)
                cv2.imwrite(os.path.join(save_dir, base + "_overlay.png"), cv2.cvtColor(ov, cv2.COLOR_RGB2BGR))

        all_metrics.append(np.mean(seed_metrics, axis=0))

    all_metrics = np.array(all_metrics, dtype=np.float32)
    return np.mean(all_metrics, axis=0), np.std(all_metrics, axis=0)

def print_result(title, mean, std):
    print(title)
    print(f"Precision mean={mean[0]:.4f} std={std[0]:.4f}")
    print(f"Recall    mean={mean[1]:.4f} std={std[1]:.4f}")
    print(f"F1        mean={mean[2]:.4f} std={std[2]:.4f}")
    print(f"IoU       mean={mean[3]:.4f} std={std[3]:.4f}")
    print(f"Time      mean={mean[4]:.4f} std={std[4]:.4f}")
    print()

def main():
    ensure_dir(OUTPUT_ROOT)
    train, val, test = get_data()

    print(len(train), len(val), len(test))

    normal_mean, normal_std = run(
        test,
        mode=None,
        save_dir=os.path.join(OUTPUT_ROOT, "test_normal"),
        save_images=True
    )
    print_result("Normal Test", normal_mean, normal_std)

    robustness = {}
    for mode in ["noise", "blur", "dark"]:
        mean, std = run(
            test,
            mode=mode,
            save_dir=os.path.join(OUTPUT_ROOT, f"test_{mode}"),
            save_images=True
        )
        robustness[mode] = (mean, std)
        print_result(f"Robustness - {mode}", mean, std)

    with open(os.path.join(OUTPUT_ROOT, "summary.txt"), "w", encoding="utf-8") as f:
        f.write("Normal Test\n")
        f.write(f"Precision mean={normal_mean[0]:.4f} std={normal_std[0]:.4f}\n")
        f.write(f"Recall    mean={normal_mean[1]:.4f} std={normal_std[1]:.4f}\n")
        f.write(f"F1        mean={normal_mean[2]:.4f} std={normal_std[2]:.4f}\n")
        f.write(f"IoU       mean={normal_mean[3]:.4f} std={normal_std[3]:.4f}\n")
        f.write(f"Time      mean={normal_mean[4]:.4f} std={normal_std[4]:.4f}\n\n")

        for mode in ["noise", "blur", "dark"]:
            mean, std = robustness[mode]
            f.write(f"Robustness - {mode}\n")
            f.write(f"Precision mean={mean[0]:.4f} std={std[0]:.4f}\n")
            f.write(f"Recall    mean={mean[1]:.4f} std={std[1]:.4f}\n")
            f.write(f"F1        mean={mean[2]:.4f} std={std[2]:.4f}\n")
            f.write(f"IoU       mean={mean[3]:.4f} std={std[3]:.4f}\n")
            f.write(f"Time      mean={mean[4]:.4f} std={std[4]:.4f}\n\n")

if __name__ == "__main__":
    main()

142 24 24
Normal Test
Precision mean=0.5303 std=0.0005
Recall    mean=0.3601 std=0.0007
F1        mean=0.4202 std=0.0004
IoU       mean=0.2932 std=0.0004
Time      mean=0.1725 std=0.0232

Robustness - noise
Precision mean=0.5318 std=0.0033
Recall    mean=0.3527 std=0.0102
F1        mean=0.4151 std=0.0088
IoU       mean=0.2882 std=0.0087
Time      mean=0.1649 std=0.0040

Robustness - blur
Precision mean=0.5324 std=0.0001
Recall    mean=0.3595 std=0.0003
F1        mean=0.4204 std=0.0003
IoU       mean=0.2922 std=0.0003
Time      mean=0.1634 std=0.0011

Robustness - dark
Precision mean=0.5274 std=0.0003
Recall    mean=0.3556 std=0.0007
F1        mean=0.4154 std=0.0006
IoU       mean=0.2888 std=0.0006
Time      mean=0.1608 std=0.0007

